In [1]:
from datetime import date

import hisepy
import pandas as pd
import polars as pl
import scanpy as sc
import json
import os
import tarfile
import session_info
import shutil

In [2]:
if os.path.isdir('results'):
    shutil.rmtree('results')

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Get the DEG app repo from Github

In [4]:
pipeline_repo = 'https://github.com/aifimmunology/sc-deg-explorer'
pipeline_sha = 'adec515c87a5b6ab183ddea9923b04bd0958efef'
pipeline_local_path = 'sc-deg-explorer'

In [5]:
clone_command = f'git config --global url."https://github.com/".insteadOf "git@github.com:" ; git clone --recurse-submodules {pipeline_repo} {pipeline_local_path}; cd {pipeline_local_path}; git reset --hard {pipeline_sha}'

In [6]:
os.system(clone_command)

HEAD is now at adec515 README.md: uv --> pixi commands


fatal: destination path 'sc-deg-explorer' already exists and is not an empty directory.


0

In [7]:
data_path = 'data'
if not os.path.isdir(data_path):
    os.makedirs(preprocess_path)

## Get DEGs

In [8]:
deg_uuid = 'afabebf4-a0fb-49f6-a842-1af3e2065714'

In [9]:
deg_file = hisepy.cache_files([deg_uuid])[0]

2026-07-29 14:36:22,718 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling cache_files
2026-07-29 14:36:27,952 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished cache_files successfully (time_elapsed=2.883s)


In [10]:
deg_file

'/home/workspace/input/1918706177/cohorts/afabebf4-a0fb-49f6-a842-1af3e2065714/tungsten-neutron-magnesium/tcell-vrd_deg_2026-07-29.csv'

In [11]:
deg = pl.read_csv(deg_file)

In [12]:
deg = deg.rename({'fg': 'Foreground', 'bg': 'Background'})

In [13]:
nz = deg.filter(pl.col('adjP') > 0)

In [14]:
min(nz['adjP'])

1e-45

In [15]:
deg.head()

Cell Type,Treatment,Timepoint,Foreground,Background,gene,n_sample,mean,logFC,nomP,adjP
str,str,str,str,str,str,i64,f64,f64,f64,f64
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""A1BG-AS1""",648,0.085197,-0.008604,0.7403701,0.999237
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAGAB""",648,0.29036,-0.054295,0.145599,0.991887
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAK1""",648,0.895006,0.014816,0.876254,0.999237
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAMDC""",648,0.2063,0.043135,0.160231,0.991887
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAMP""",648,0.068101,-0.010012,0.7332528,0.999237


### Fix 0 adjP values

In [16]:
deg = deg.with_columns(
    pl.when(pl.col("adjP") == 0)
    .then(1e-100)
    .otherwise(pl.col("adjP"))
    .alias("adjP")
)

In [17]:
deg.filter(
    pl.col('Foreground') == 'Dexamethasone',
    pl.col('gene') == 'FKBP5'
)

Cell Type,Treatment,Timepoint,Foreground,Background,gene,n_sample,mean,logFC,nomP,adjP
str,str,str,str,str,str,i64,f64,f64,f64,f64
"""CD4 CM""","""Dexamethasone""","""T4""","""Dexamethasone""","""DMSO""","""FKBP5""",981,2.04632,0.978923,0.0,1.0000e-100
"""CD4 EM""","""Dexamethasone""","""T4""","""Dexamethasone""","""DMSO""","""FKBP5""",401,1.384819,1.2258573,0.0,1.0000e-100
"""CD4 Naive""","""Dexamethasone""","""T4""","""Dexamethasone""","""DMSO""","""FKBP5""",1739,2.380753,0.654011,0.0,1.0000e-100
"""CD4 Treg""","""Dexamethasone""","""T4""","""Dexamethasone""","""DMSO""","""FKBP5""",210,1.461995,0.6112167,0.000003,0.000722
"""CD8 Memory""","""Dexamethasone""","""T4""","""Dexamethasone""","""DMSO""","""FKBP5""",319,1.315594,1.6016601,0.0,1.0000e-100
…,…,…,…,…,…,…,…,…,…,…
"""CD4 EM""","""Dexamethasone""","""T24""","""Dexamethasone""","""DMSO""","""FKBP5""",401,1.274532,1.5317633,0.0,1.0000e-100
"""CD4 Naive""","""Dexamethasone""","""T24""","""Dexamethasone""","""DMSO""","""FKBP5""",1739,1.354799,0.99131,0.0,1.0000e-100
"""CD4 Treg""","""Dexamethasone""","""T24""","""Dexamethasone""","""DMSO""","""FKBP5""",210,1.228902,1.3687241,1.1888e-31,9.7601e-28


In [18]:
deg_preprocess_file = f'{data_path}/tcell-vrd_deg.csv'
deg.write_csv(deg_preprocess_file)

## Get GSEA gene sets

In [19]:
gs_uuid = '592f79bd-b8c2-456d-ae0c-bc5259e5af9d'
gs_file = hisepy.cache_files([gs_uuid])[0]

2026-07-29 14:36:28,329 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling cache_files
2026-07-29 14:36:33,626 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished cache_files successfully (time_elapsed=2.993s)


In [20]:
gs_name = os.path.basename(gs_file)

In [21]:
shutil.copyfile(gs_file, f'data/{gs_name}')

'data/hallmark_reactome-v87_gene_sets_2026-07-29.json'

## Get GSEA results

In [22]:
hallmark_uuid = '8d61354a-c7b4-4b73-81c9-54287d020284'
reactome_uuid = 'ebc68238-1c39-4cf5-b7b4-bfde7b1f1afd'

In [23]:
hallmark_file = hisepy.cache_files([hallmark_uuid])[0]
reactome_file = hisepy.cache_files([reactome_uuid])[0]

2026-07-29 14:36:33,738 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling cache_files
2026-07-29 14:36:38,543 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished cache_files successfully (time_elapsed=2.508s)
2026-07-29 14:36:38,544 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling cache_files
2026-07-29 14:36:43,370 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished cache_files successfully (time_elapsed=2.515s)


In [24]:
hallmark = pl.read_csv(hallmark_file, separator = '\t')
reactome = pl.read_csv(reactome_file, separator = '\t')

In [25]:
hallmark = hallmark.with_columns(
    pl.lit('Hallmark').alias('Source')
).rename({'fg': 'Foreground', 'bg': 'Background'})
reactome = reactome.with_columns(
    pl.lit('Reactome').alias('Source')
).rename({'fg': 'Foreground', 'bg': 'Background'})

In [26]:
gsea = pl.concat([hallmark, reactome])
gsea = gsea.rename({
    'pathway':'Pathway',
    'leading_edge_genes': 'leadingEdge'
})

In [27]:
gsea_preprocess_file = f'{data_path}/tcell-vrd_gsea.tsv'
gsea.write_csv(gsea_preprocess_file, separator = '\t')

## Run preprocessing

In [28]:
pp_path = f'{pipeline_local_path}/preprocess.py'
pp_command = f'pixi run python {pp_path} --config deg-configs/tcell-vrd.json'

In [29]:
os.system(pp_command)

Writing to results/deg/deg.parquet
Writing to results/deg/meta.parquet
Writing to results/deg/summaries.parquet
Writing to results/deg/adjp.parquet
Writing to results/deg/t-adjp.parquet
Writing to results/deg/log2fc.parquet
Writing to results/deg/t-log2fc.parquet
Writing to results/deg/means.parquet
Writing to results/gsea/results.parquet
Writing to results/gsea/meta.parquet
Writing to results/gsea/summaries.parquet


0

## Fix labels

In [30]:
gsea_meta = pl.read_parquet('results/gsea/meta.parquet')
gsea_meta = gsea_meta.with_columns(
    pl.col('Pathway').alias('Pathway Label')
)
gsea_meta.write_parquet('results/gsea/meta.parquet')

In [31]:
gsea_res = pl.read_parquet('results/gsea/results.parquet')
gsea_res = gsea_res.with_columns(
    pl.col('Pathway').alias('Pathway Label')
)
gsea_res.write_parquet('results/gsea/results.parquet')

## Tar outputs and save to HISE

In [32]:
out_tar = 'tcell-vrd_deg_vis_{d}.tar'.format(d = date.today())
with tarfile.open(out_tar, "w") as tar:
    tar.add('results')

In [33]:
search_id = element_id()
search_id

'californium-manganese-carbon'

In [34]:
session_info.show()

/home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/python3.13/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
/home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/python3.13/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)


In [35]:
hisepy.upload_files(
    files = [out_tar],
    study_space_id = '40df6403-29f0-4b45-ab7d-f46d420c422e',
    title = 'T cell VRd DEG files formatted for DEG Explorer {d}'.format(d = date.today()),
    input_file_ids = [deg_uuid],
    destination = search_id
)

2026-07-29 14:36:48,455 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling upload_files
2026-07-29 14:36:48,456 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling upload_files_internal


Please provide input of comma separated sample ids or sample kit guids for the files being uploaded. If you do not have any samples, press enter:  


2026-07-29 14:38:06,244 INFO [hisepy.logging:185] logging 18554 136284826875712 Calling get_default_store
2026-07-29 14:38:09,195 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished get_default_store successfully (time_elapsed=0.563s)
2026-07-29 14:38:20,218 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished upload_files_internal successfully (time_elapsed=89.362s)
2026-07-29 14:38:22,413 INFO [hisepy.logging:228] logging 18554 136284826875712 Finished upload_files successfully (time_elapsed=91.764s)


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '66bdde08-2203-41f2-82d3-206b23d7c435',
 'ProcessId': 'b74fa7d1-6bab-4e4c-9eaa-71e2db65d572',
 'WorkflowId': '5213ac75-0646-4e43-b80b-8ef94355595f',
 'FileIds': ['b5f6de81-5ee3-4ed7-b737-e273065ec23c']}